# Imports


In [ ]:
from __future__ import annotations

import gzip
import math
import struct
from pathlib import Path
from typing import Iterable

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, random_split


# Constants


In [ ]:
RANDOM_SEED = 42


In [ ]:
DATA_DIR: Path = Path("/home/linkezio/Datasets/MNIST")


In [ ]:
MODELS_DIR: Path = Path("/home/linkezio/Projects/Efficient-Polling-Based-Learning-Rate-Optimization-for-Neural-Networks/models")


# Configs


## Seeds


In [ ]:
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


## Device


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)


# Data


## IDX helpers


In [ ]:
def _open_maybe_gzip(path: Path):
    return gzip.open(path, "rb") if path.suffix == ".gz" else path.open("rb")


def _read_idx(path: Path) -> np.ndarray:
    with _open_maybe_gzip(path) as f:
        magic, = struct.unpack(">I", f.read(4))
        if magic not in {2049, 2051}:
            raise ValueError(f"Arquivo IDX inválido: {path} (magic={magic})")
        dims, = struct.unpack(">I", f.read(4))
        shape = tuple(struct.unpack(">I", f.read(4))[0] for _ in range(dims))
        data = f.read()

    arr = np.frombuffer(data, dtype=np.uint8)
    return arr.reshape(shape)


def find_mnist_files(data_dir: Path) -> dict[str, Path]:
    candidates = [p for p in data_dir.glob("*") if p.is_file()]
    names = {p.name: p for p in candidates}

    def pick(prefixes: Iterable[str]) -> Path:
        for p in candidates:
            lower = p.name.lower()
            if any(lower.startswith(pref) for pref in prefixes) and ("idx" in lower or "ubyte" in lower):
                return p
        raise FileNotFoundError(f"Não achei arquivos MNIST em {data_dir}. Arquivos encontrados: {sorted(names)}")

    return {
        "train_images": pick(["train-images", "train_images", "train-images-idx3"]),
        "train_labels": pick(["train-labels", "train_labels", "train-labels-idx1"]),
        "test_images": pick(["t10k-images", "test-images", "t10k_images", "t10k-images-idx3"]),
        "test_labels": pick(["t10k-labels", "test-labels", "t10k_labels", "t10k-labels-idx1"]),
    }


## Dataset Class


In [ ]:
class MNISTDataset(Dataset):
    def __init__(
        self,
        data_dir: Path,
        train: bool,
        mean: torch.Tensor | None = None,
        std: torch.Tensor | None = None,
    ):
        files = find_mnist_files(data_dir)
        if train:
            images_path, labels_path = files["train_images"], files["train_labels"]
        else:
            images_path, labels_path = files["test_images"], files["test_labels"]

        images = _read_idx(images_path)
        labels = _read_idx(labels_path)
        if images.ndim != 3:
            raise ValueError(f"Imagens MNIST esperadas (N,H,W). Recebido: {images.shape}")
        if labels.ndim != 1:
            raise ValueError(f"Labels MNIST esperados (N,). Recebido: {labels.shape}")
        if images.shape[0] != labels.shape[0]:
            raise ValueError("N de imagens != N de labels")

        self.images = torch.from_numpy(images).float().unsqueeze(1)  # (N,1,28,28), 0–255
        self.labels = torch.from_numpy(labels).long()

        if (mean is None) ^ (std is None):
            raise ValueError("Passe `mean` e `std` juntos, ou nenhum dos dois.")
        self.mean = mean
        self.std = std

    def __len__(self) -> int:
        return int(self.labels.shape[0])

    def __getitem__(self, idx: int):
        x = self.images[idx]
        if self.mean is not None:
            x = (x - self.mean) / self.std
        y = self.labels[idx]
        return x, y


## Calculate mean and std for normalize later


In [ ]:

def compute_mean_std(dataset, batch_size=512):
    """Média e desvio padrão por canal sobre todos os pixels."""
    loader_mean = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    channel_sum = None
    n_pixels = 0
    for images, _ in loader_mean:
        b, c, h, w = images.shape
        if channel_sum is None:
            channel_sum = torch.zeros(c, dtype=torch.float64)
        channel_sum += images.double().sum(dim=(0, 2, 3))
        n_pixels += b * h * w

    mean = (channel_sum / n_pixels).float()

    loader_var = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    sum_sq = None
    for images, _ in loader_var:
        b, c, h, w = images.shape
        if sum_sq is None:
            sum_sq = torch.zeros(c, dtype=torch.float64)
        diff = images.double() - mean.view(1, c, 1, 1).double()
        sum_sq += (diff * diff).sum(dim=(0, 2, 3))

    var = (sum_sq / n_pixels).float()
    std = torch.sqrt(var)
    std = torch.clamp(std, min=1e-8)
    return mean, std


In [ ]:
train_for_stats = MNISTDataset(DATA_DIR, train=True)

mnist_mean, mnist_std = compute_mean_std(train_for_stats)

mnist_mean = mnist_mean.view(1, 1, 1)
mnist_std = mnist_std.view(1, 1, 1)

print("mean (1 canal):", mnist_mean.squeeze().tolist())
print("std  (1 canal):", mnist_std.squeeze().tolist())


## Train, Validation, Test Split


In [ ]:
class DataLoaderHyperparameters:
    batch_size: int = 128
    val_fraction: float = 0.1
    num_workers: int = 0  # Jupyter: use 0 (workers não acham classes em __main__).

data_loader_hyperparameters = DataLoaderHyperparameters()


In [ ]:
full_train = MNISTDataset(DATA_DIR, train=True, mean=mnist_mean, std=mnist_std)
test_ds = MNISTDataset(DATA_DIR, train=False, mean=mnist_mean, std=mnist_std)

val_size = max(1, int(len(full_train) * data_loader_hyperparameters.val_fraction))
train_size = len(full_train) - val_size

train_ds, val_ds = random_split(
    full_train,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(RANDOM_SEED),
)

train_loader = DataLoader(
    train_ds,
    batch_size=data_loader_hyperparameters.batch_size,
    shuffle=True,
    num_workers=data_loader_hyperparameters.num_workers,
    pin_memory=(DEVICE == "cuda"),
)
val_loader = DataLoader(
    val_ds,
    batch_size=data_loader_hyperparameters.batch_size,
    shuffle=False,
    num_workers=data_loader_hyperparameters.num_workers,
    pin_memory=(DEVICE == "cuda"),
)
test_loader = DataLoader(
    test_ds,
    batch_size=data_loader_hyperparameters.batch_size,
    shuffle=False,
    num_workers=data_loader_hyperparameters.num_workers,
    pin_memory=(DEVICE == "cuda"),
)

len(train_ds), len(val_ds), len(test_ds)


# Model


## Hyperparameters


In [ ]:
class ModelHyperparameters:
    batch_size: int = 128
    epochs: int = 5
    lr: float = 1e-3
    weight_decay: float = 0.0
    num_workers: int = 0  # Jupyter

model_hyperparameters = ModelHyperparameters()


## Model Class


In [ ]:
class SimpleMNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        return self.classifier(x)


# Training


## Metric Functions


In [ ]:
def accuracy(logits: torch.Tensor, y: torch.Tensor) -> float:
    preds = logits.argmax(dim=1)
    return (preds == y).float().mean().item()


In [ ]:
loss_fn = nn.CrossEntropyLoss()


## Epoch Functions


In [ ]:
@torch.inference_mode()
def eval_epoch(model: nn.Module, loader: DataLoader, loss_fn: nn.Module) -> tuple[float, float]:
    model.eval()
    losses = []
    accs = []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        logits = model(x)
        losses.append(loss_fn(logits, y).item())
        accs.append(accuracy(logits, y))
    return float(np.mean(losses)), float(np.mean(accs))


def train_epoch(model: nn.Module, loader: DataLoader, optim: torch.optim.Optimizer, loss_fn: nn.Module) -> tuple[float, float]:
    model.train()
    losses = []
    accs = []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optim.zero_grad(set_to_none=True)
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        optim.step()

        losses.append(loss.item())
        accs.append(accuracy(logits.detach(), y))
    return float(np.mean(losses)), float(np.mean(accs))


In [ ]:
model = SimpleMNISTCNN().to(DEVICE)
optim = torch.optim.Adam(
    model.parameters(),
    lr=model_hyperparameters.lr,
    weight_decay=model_hyperparameters.weight_decay,
)


In [ ]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
}

best_val_acc = -math.inf
model_path = MODELS_DIR / "mnist_best.pt"

for epoch in range(1, model_hyperparameters.epochs + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, optim, loss_fn)
    va_loss, va_acc = eval_epoch(model, val_loader, loss_fn)

    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_loss"].append(va_loss)
    history["val_acc"].append(va_acc)

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save(model.state_dict(), model_path)

    print(
        f"epoch {epoch:02d}/{model_hyperparameters.epochs} | "
        f"train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
        f"val loss {va_loss:.4f} acc {va_acc:.4f}"
    )

print("best val acc:", best_val_acc, "saved:", str(model_path))


# Test


In [ ]:
model.load_state_dict(torch.load(MODELS_DIR / "mnist_best.pt", map_location=DEVICE))

test_loss, test_acc = eval_epoch(model, test_loader, loss_fn)

print(f"test loss {test_loss:.4f} | test acc {test_acc:.4f}")
